# Representación del conocimiento

Cómo se guarda el conocimiento decide qué puede concluir un programa a partir de él. Dos respuestas clásicas organizan los conceptos en jerarquías, de forma que lo que es cierto de un concepto general lo heredan los específicos.

## Redes semánticas

Una **red semántica** es un grafo de conceptos unidos por relaciones etiquetadas: *un pingüino es un ave*, *un ave es un animal*. Una propiedad indicada para un concepto la **hereda** todo lo que está por debajo, **salvo que un concepto más específico diga otra cosa**: las aves vuelan, pero los pingüinos no. El razonamiento que un hecho posterior y más específico puede revocar se llama razonamiento *por defecto* o *no monótono*, y aquí la regla es sencilla: gana la respuesta más cercana subiendo por la cadena `is_a`.

Cada hecho es un triple, sujeto, relación y valor, la misma forma que los **grafos de conocimiento** actuales, como Wikidata.

In [ ]:
animal_facts = [
    ("bird", "is_a", "animal"), ("mammal", "is_a", "animal"),
    ("penguin", "is_a", "bird"), ("sparrow", "is_a", "bird"), ("dog", "is_a", "mammal"),
    ("animal", "breathes", True),
    ("bird", "flies", True), ("bird", "covering", "feathers"),
    ("mammal", "covering", "fur"),
    ("penguin", "flies", False), ("penguin", "swims", True),
]


def look_up(facts, concept, relation):
    for subject, fact_relation, value in facts:
        if subject == concept and fact_relation == relation:
            return value
    parents = [value for subject, fact_relation, value in facts if subject == concept and fact_relation == "is_a"]
    for parent in parents:
        value = look_up(facts, parent, relation)
        if value is not None:
            return value
    return None

In [ ]:
for concept in ["sparrow", "penguin", "dog"]:
    print(f"{concept:8} flies: {look_up(animal_facts, concept, 'flies')}, "
          f"covering: {look_up(animal_facts, concept, 'covering')}, "
          f"breathes: {look_up(animal_facts, concept, 'breathes')}")

sparrow  flies: True, covering: feathers, breathes: True
penguin  flies: False, covering: feathers, breathes: True
dog      flies: None, covering: fur, breathes: True


El gorrión hereda *vuela* de ave, el hecho propio del pingüino lo anula, y todo animal hereda *respira* de la cima de la jerarquía. El perro no vuela, pero la red no puede decirlo: `None` significa *desconocido*, no *falso*.

## Marcos (*frames*)

Un **marco** (*frame*; Marvin Minsky, 1974) describe un objeto o una situación estereotípicos como un conjunto de **ranuras** (*slots*) con valores, y hereda las ranuras que no rellena de un marco más general. Si eso suena a una clase con atributos y una clase padre, es porque lo es: los marcos son una de las ideas detrás de la **programación orientada a objetos**.

In [ ]:
vehicle = {"parent": None, "wheels": 4, "engine": True, "doors": 4}
car = {"parent": vehicle, "fuel": "petrol"}
sports_car = {"parent": car, "doors": 2, "top_speed_kmh": 250}
motorcycle = {"parent": vehicle, "wheels": 2, "doors": 0}


def slot(frame, name):
    while frame is not None:
        if name in frame:
            return frame[name]
        frame = frame["parent"]
    return None


for name, frame in [("car", car), ("sports car", sports_car), ("motorcycle", motorcycle)]:
    print(f"{name:11} wheels: {slot(frame, 'wheels')}, doors: {slot(frame, 'doors')}, engine: {slot(frame, 'engine')}")

car         wheels: 4, doors: 4, engine: True
sports car  wheels: 4, doors: 2, engine: True
motorcycle  wheels: 2, doors: 0, engine: True


Las clases de Python hacen la misma búsqueda, a través de sus atributos de clase y la cadena de herencia:

In [ ]:
class Vehicle:
    wheels, engine, doors = 4, True, 4


class Car(Vehicle):
    fuel = "petrol"


class SportsCar(Car):
    doors, top_speed_kmh = 2, 250


class Motorcycle(Vehicle):
    wheels, doors = 2, 0


print("SportsCar doors:", SportsCar.doors, "| wheels:", SportsCar.wheels, "| fuel:", SportsCar.fuel)
print("Motorcycle doors:", Motorcycle.doors, "| engine:", Motorcycle.engine)

SportsCar doors: 2 | wheels: 4 | fuel: petrol
Motorcycle doors: 0 | engine: True


El diseño orientado a objetos del [nivel 3 del tres en raya](https://github.com/avidaldo/python-tic-tac-toe/tree/master/03_oop) usa el mismo mecanismo: cada tipo de jugador hereda de una clase `Player` y rellena su propia forma de elegir una jugada.